In [1]:
# CELL 1: setup, data loading, tokenisation, and class weights

import os
import json
import time
import warnings
from collections import Counter

import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from transformers.utils import logging as hf_logging



# keep notebook output clean i.e stop unnecessary printing errors
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*Some weights of .* were not initialized.*")
hf_logging.set_verbosity_error()


# core config
model_id = "thomas-sounack/BioClinical-ModernBERT-base"

train_path = "../Preprocessing+baseline/data/pqal_fold0/train_set.json"
dev_path = "../Preprocessing+baseline/data/pqal_fold0/dev_set.json"

label_to_id = {"no": 0, "maybe": 1, "yes": 2}
id_to_label = {0: "no", 1: "maybe", 2: "yes"}

# speed-focused settings
train_batch_size = 8
eval_batch_size = 8
grad_accum_steps = 2


# tokenisation 
tokenise_num_proc = 4
dataloader_workers = 2

num_epochs = 8
early_stopping = 2

# Initially introduced regularisation using weight_decay = 0.01 and warmup_ratio = 0.1 but decided agaaint that due to poorer performance
weight_decay = 0.0
warmup_ratio = 0.0

max_grad_norm = 1.0

tokenizer = AutoTokenizer.from_pretrained(model_id)


def load_pubmed_split(path):
    # Load one PubMedQA split into a Hugging Face Dataset.
    with open(path, "r") as f:
        raw = json.load(f)

    rows = []
    for example_id, row in raw.items():
        rows.append(
            {
                "ID": example_id,
                "QUESTION": row["QUESTION"],
                "CONTEXTS": " ".join(row["CONTEXTS"]),
                "final_decision": row["final_decision"].strip().lower(),
            }
        )

    return Dataset.from_list(rows)


def add_label(example):
    # Map string labels to integer ids.
    example["labels"] = label_to_id[example["final_decision"]]
    return example


def tokenize_batch(batch):
    """
    Tokenise question + context pairs.

    We keep the question intact and only truncate the context if needed: truncation="only_second", second is context!.
    This is safer because the question is short and most critical.
    """
    return tokenizer(
        batch["QUESTION"],
        batch["CONTEXTS"],
        truncation="only_second",
        max_length=512,
    )


def print_divider(char="=", width=88):
    print(char * width)


def print_dataset_summary(dataset_dict, train_counts, class_weights_tensor):
    # Summary of the dataset and weighting setup.
    
    print_divider()
    print("DATA SUMMARY")
    print_divider()
    print(f"Train examples : {len(dataset_dict['train'])}")
    print(f"Dev examples   : {len(dataset_dict['validation'])}")
    print()
    print("Train label counts")
    print(f"  no    : {train_counts[0]}")
    print(f"  maybe : {train_counts[1]}")
    print(f"  yes   : {train_counts[2]}")
    print()
    print("Class weights used for weighted loss")
    print(f"  no    : {class_weights_tensor[0].item():.4f}")
    print(f"  maybe : {class_weights_tensor[1].item():.4f}")
    print(f"  yes   : {class_weights_tensor[2].item():.4f}")
    print_divider()



# load raw fold0 data
fold0_train = load_pubmed_split(train_path)
fold0_dev = load_pubmed_split(dev_path)

fold0_dataset = DatasetDict({
        "train": fold0_train,
        "validation": fold0_dev,
    })

# add labels first
fold0_dataset = fold0_dataset.map(add_label)

# tokenise
fold0_tokenized_dataset = fold0_dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=tokenise_num_proc,
)

# keep only what the trainer actually needs
fold0_tokenized_dataset = fold0_tokenized_dataset.remove_columns(
    ["ID", "QUESTION", "CONTEXTS", "final_decision"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# class weights and sample weights

train_label_ids = fold0_tokenized_dataset["train"]["labels"]
train_counts = Counter(train_label_ids)

total_train = len(train_label_ids)
num_classes = 3

class_weights = torch.tensor(
    [
        total_train / (num_classes * train_counts[0]),
        total_train / (num_classes * train_counts[1]),
        total_train / (num_classes * train_counts[2]),
    ],
    dtype=torch.float,
)

sample_weights = torch.tensor(
    [class_weights[label].item() for label in train_label_ids],
    dtype=torch.double,
)

print_dataset_summary(fold0_tokenized_dataset, train_counts, class_weights)


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/450 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

DATA SUMMARY
Train examples : 450
Dev examples   : 50

Train label counts
  no    : 152
  maybe : 49
  yes   : 249

Class weights used for weighted loss
  no    : 0.9868
  maybe : 3.0612
  yes   : 0.6024


In [2]:
# CELL 2: metrics, trainer, and run helpers


def compute_metrics(eval_pred):
    # Returns metrics.
    
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    per_class_f1 = f1_score(
        labels,
        preds,
        average=None,
        labels=[0, 1, 2],
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_no": per_class_f1[0],
        "f1_maybe": per_class_f1[1],
        "f1_yes": per_class_f1[2],
    }


class BetterTrainer(Trainer):
    """
    custom Trainer that supports:
        -  weighted loss
        -  weighted sampler
    """

    def __init__(
        self,
        *args,
        class_weights=None,
        sample_weights=None,
        use_weighted_loss=False,
        use_weighted_sampler=False,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.sample_weights = sample_weights
        self.use_weighted_loss = use_weighted_loss
        self.use_weighted_sampler = use_weighted_sampler

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.use_weighted_loss:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device)
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss()

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if not self.use_weighted_sampler:
            return super().get_train_dataloader()

        sampler = WeightedRandomSampler(
            weights=self.sample_weights,
            num_samples=len(self.sample_weights),
            replacement=True,
        )

        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=False,
        )


def make_model():
    # Create a fresh classification model each run. 
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=3,
        id2label=id_to_label,
        label2id=label_to_id,
    )


def build_training_args(run_name, seed, learning_rate, gradient_accumulation_steps):
    # Build TrainingArguments.

    return TrainingArguments(
        output_dir=f"./runs/{run_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="no",
        disable_tqdm=True,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        max_grad_norm=max_grad_norm,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        seed=seed,
        data_seed=seed,
        dataloader_num_workers=dataloader_workers,
        dataloader_pin_memory=False,
    )


def short_name(name, max_len=36):
    # For readability
    return name if len(name) <= max_len else name[: max_len - 3] + "..."


def clean_eval_metrics(metrics):
    return {
        "accuracy": metrics["eval_accuracy"],
        "macro_f1": metrics["eval_macro_f1"],
        "f1_no": metrics["eval_f1_no"],
        "f1_maybe": metrics["eval_f1_maybe"],
        "f1_yes": metrics["eval_f1_yes"],
    }


def print_stage_header(stage_title, stage_explainer):
    print()
    print_divider()
    print(stage_title.upper())
    print_divider()
    print(stage_explainer)
    print_divider()


def print_run_summary(result):
    m = result["final_metrics"]

    print(f"\nFinished: {result['run_name']}")
    print(
        f"  setup      : seed={result['seed']} | lr={result['learning_rate']} | "
        f"batch={train_batch_size} | ga={result['gradient_accumulation_steps']} | "
        f"weighted_loss={result['use_weighted_loss']} | weighted_sampler={result['use_weighted_sampler']}"
    )
    print(f"  runtime    : {result['runtime_minutes']:.2f} min")
    print(f"  accuracy   : {m['accuracy']:.4f}")
    print(f"  macro_f1   : {m['macro_f1']:.4f}")
    print(f"  f1_no      : {m['f1_no']:.4f}")
    print(f"  f1_maybe   : {m['f1_maybe']:.4f}")
    print(f"  f1_yes     : {m['f1_yes']:.4f}")


def print_results_table(results, title):
    # Print a table for a whole stage.
    print()
    print_divider()
    print(title)
    print_divider()

    header = (
        f"{'run_name':<38}"
        f"{'macro_f1':>10}"
        f"{'acc':>10}"
        f"{'f1_maybe':>12}"
        f"{'time(min)':>12}"
    )
    print(header)
    print("-" * len(header))

    for r in results:
        m = r["final_metrics"]
        print(
            f"{short_name(r['run_name'], 38):<38}"
            f"{m['macro_f1']:>10.4f}"
            f"{m['accuracy']:>10.4f}"
            f"{m['f1_maybe']:>12.4f}"
            f"{r['runtime_minutes']:>12.2f}"
        )

    print_divider()


def print_confusion_matrix_nicely(cm):
    print("\nConfusion matrix (rows = true label, columns = predicted label)")
    print(f"{'':<10}{'no':>8}{'maybe':>10}{'yes':>8}")
    labels = ["no", "maybe", "yes"]
    for label_name, row in zip(labels, cm):
        print(f"{label_name:<10}{row[0]:>8}{row[1]:>10}{row[2]:>8}")


def run_experiment(
    run_name,
    seed,
    learning_rate,
    gradient_accumulation_steps,
    use_weighted_loss,
    use_weighted_sampler,
    collect_predictions=False,
):

    start_time = time.perf_counter()

    trainer = BetterTrainer(
        model_init=make_model,
        args=build_training_args(
            run_name=run_name,
            seed=seed,
            learning_rate=learning_rate,
            gradient_accumulation_steps=gradient_accumulation_steps,
        ),
        train_dataset=fold0_tokenized_dataset["train"],
        eval_dataset=fold0_tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        sample_weights=sample_weights,
        use_weighted_loss=use_weighted_loss,
        use_weighted_sampler=use_weighted_sampler,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping)],
    )

    trainer.train()
    raw_final_metrics = trainer.evaluate()
    runtime_minutes = (time.perf_counter() - start_time) / 60.0

    final_metrics = clean_eval_metrics(raw_final_metrics)

    result = {
        "run_name": run_name,
        "seed": seed,
        "learning_rate": learning_rate,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "use_weighted_loss": use_weighted_loss,
        "use_weighted_sampler": use_weighted_sampler,
        "runtime_minutes": runtime_minutes,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "epochs_ran": trainer.state.epoch,
        "final_metrics": final_metrics,
    }

    if collect_predictions:
        pred_output = trainer.predict(fold0_tokenized_dataset["validation"])
        preds = np.argmax(pred_output.predictions, axis=-1)
        labels = pred_output.label_ids
        result["confusion_matrix"] = confusion_matrix(labels, preds, labels=[0, 1, 2])

    print_run_summary(result)
    return result

In [3]:
# CELL 3: Stage 1 - quick sweep of training recipe


print_stage_header(
    stage_title="Stage 1",
    stage_explainer=(
    "This stage is a cheap first pass on fold0 using one seed only. "
    "The aim is not to find the final best model yet, but to identify the best overall training recipe "
    "before spending more time on tuning. By training recipe, we mean the broad setup choices that control "
    "how the model learns from the imbalanced data, rather than small parameter tweaks. "
    "Here we compare the main options we considered: weighted loss only, weighted loss plus weighted sampler, "
    "and weighted sampler only. Weighted loss means mistakes on rare classes, especially 'maybe', are penalised more heavily. "
    "Weighted sampling means the model sees minority-class examples more often during training. "
    "We tested these because our earlier results showed that naive fine-tuning was collapsing toward the majority class "
    "and failing badly on 'maybe'. So this stage is really about asking what kind of correction works best for the imbalance problem: "
    "changing how errors are penalised, changing what the model sees in each batch, or combining both. "
    "We keep this stage cheap by using only one seed, because at this point we only need a clear direction, not a final stability claim."
),
)

stage1_configs = [
    {
        "run_name": "stage1_loss_only_lr1e5_seed42",
        "seed": 42,
        "learning_rate": 1e-5,
        "gradient_accumulation_steps": grad_accum_steps,
        "use_weighted_loss": True,
        "use_weighted_sampler": False,
    },
    {
        "run_name": "stage1_loss_plus_sampler_lr1e5_seed42",
        "seed": 42,
        "learning_rate": 1e-5,
        "gradient_accumulation_steps": grad_accum_steps,
        "use_weighted_loss": True,
        "use_weighted_sampler": True,
    },
    {
        "run_name": "stage1_loss_plus_sampler_lr5e6_seed42",
        "seed": 42,
        "learning_rate": 5e-6,
        "gradient_accumulation_steps": grad_accum_steps,
        "use_weighted_loss": True,
        "use_weighted_sampler": True,
    },
    {
        "run_name": "stage1_sampler_only_lr1e5_seed42",
        "seed": 42,
        "learning_rate": 1e-5,
        "gradient_accumulation_steps": grad_accum_steps,
        "use_weighted_loss": False,
        "use_weighted_sampler": True,
    },
]

stage1_results = []

for i, cfg in enumerate(stage1_configs, start=1):
    print(f"\nRun {i}/{len(stage1_configs)}")
    stage1_results.append(run_experiment(**cfg))

print_results_table(stage1_results, "Stage 1 summary")

best_stage1 = max(stage1_results, key=lambda x: x["final_metrics"]["macro_f1"])

print("\nBest Stage 1 setup")
print(f"  run_name          : {best_stage1['run_name']}")
print(f"  macro_f1          : {best_stage1['final_metrics']['macro_f1']:.4f}")
print(f"  accuracy          : {best_stage1['final_metrics']['accuracy']:.4f}")
print(f"  f1_maybe          : {best_stage1['final_metrics']['f1_maybe']:.4f}")
print(f"  weighted_loss     : {best_stage1['use_weighted_loss']}")
print(f"  weighted_sampler  : {best_stage1['use_weighted_sampler']}")


STAGE 1
This stage is a cheap first pass on fold0 using one seed only. The aim is not to find the final best model yet, but to identify the best overall training recipe before spending more time on tuning. By training recipe, we mean the broad setup choices that control how the model learns from the imbalanced data, rather than small parameter tweaks. Here we compare the main options we considered: weighted loss only, weighted loss plus weighted sampler, and weighted sampler only. Weighted loss means mistakes on rare classes, especially 'maybe', are penalised more heavily. Weighted sampling means the model sees minority-class examples more often during training. We tested these because our earlier results showed that naive fine-tuning was collapsing toward the majority class and failing badly on 'maybe'. So this stage is really about asking what kind of correction works best for the imbalance problem: changing how errors are penalised, changing what the model sees in each batch, or co

In [4]:
# CELL 4: Stage 2 - tune the best Stage 1 recipe


print_stage_header(
    stage_title="Stage 2",
    stage_explainer=(
        "Now we keep the best Stage 1 training recipe fixed and do a small "
        "learning-rate sweep. This keeps the search controlled and avoids wasting runs."
    ),
)

stage2_learning_rates = [1e-5, 8e-6, 6e-6]

stage2_configs = []
for lr in stage2_learning_rates:
    lr_name = str(lr).replace("-", "").replace(".", "")
    stage2_configs.append(
        {
            "run_name": f"stage2_bestrecipe_lr_{lr_name}_seed42",
            "seed": 42,
            "learning_rate": lr,
            "gradient_accumulation_steps": grad_accum_steps,
            "use_weighted_loss": best_stage1["use_weighted_loss"],
            "use_weighted_sampler": best_stage1["use_weighted_sampler"],
        }
    )

stage2_results = []

for i, cfg in enumerate(stage2_configs, start=1):
    print(f"\nRun {i}/{len(stage2_configs)}")
    stage2_results.append(run_experiment(**cfg))

print_results_table(stage2_results, "Stage 2 summary")

best_stage2 = max(stage2_results, key=lambda x: x["final_metrics"]["macro_f1"])

print("\nBest Stage 2 setup")
print(f"  run_name          : {best_stage2['run_name']}")
print(f"  macro_f1          : {best_stage2['final_metrics']['macro_f1']:.4f}")
print(f"  accuracy          : {best_stage2['final_metrics']['accuracy']:.4f}")
print(f"  f1_maybe          : {best_stage2['final_metrics']['f1_maybe']:.4f}")
print(f"  learning_rate     : {best_stage2['learning_rate']}")
print(f"  weighted_loss     : {best_stage2['use_weighted_loss']}")
print(f"  weighted_sampler  : {best_stage2['use_weighted_sampler']}")

best_stage2_template = {
    "learning_rate": best_stage2["learning_rate"],
    "gradient_accumulation_steps": best_stage2["gradient_accumulation_steps"],
    "use_weighted_loss": best_stage2["use_weighted_loss"],
    "use_weighted_sampler": best_stage2["use_weighted_sampler"],
}


STAGE 2
Now we keep the best Stage 1 training recipe fixed and do a small learning-rate sweep. This keeps the search controlled and avoids wasting runs.

Run 1/3
{'eval_loss': 1.1196167469024658, 'eval_accuracy': 0.36, 'eval_macro_f1': 0.2900400228702115, 'eval_f1_no': 0.16, 'eval_f1_maybe': 0.18181818181818182, 'eval_f1_yes': 0.5283018867924528, 'eval_runtime': 20.2038, 'eval_samples_per_second': 2.475, 'eval_steps_per_second': 0.346, 'epoch': 1.0}
{'eval_loss': 1.0772384405136108, 'eval_accuracy': 0.42, 'eval_macro_f1': 0.36963629622139077, 'eval_f1_no': 0.1935483870967742, 'eval_f1_maybe': 0.36363636363636365, 'eval_f1_yes': 0.5517241379310345, 'eval_runtime': 20.2139, 'eval_samples_per_second': 2.474, 'eval_steps_per_second': 0.346, 'epoch': 2.0}
{'eval_loss': 1.0827285051345825, 'eval_accuracy': 0.42, 'eval_macro_f1': 0.3895800106326422, 'eval_f1_no': 0.24242424242424243, 'eval_f1_maybe': 0.4, 'eval_f1_yes': 0.5263157894736842, 'eval_runtime': 20.451, 'eval_samples_per_second': 2

In [5]:
# CELL 5: Stage 3 - 3-seed stability check and final summary

print_stage_header(
    stage_title="Stage 3",
    stage_explainer=(
        "Final stability check. We take the best Stage 2 config and run it "
        "across 3 seeds so we can see whether it is genuinely stable or just lucky."
    ),
)

stage3_results = []

for i, seed in enumerate([42, 52, 62], start=1):
    cfg = {
        "run_name": f"stage3_bestconfig_seed_{seed}",
        "seed": seed,
        **best_stage2_template,
    }

    print(f"\nRun {i}/3")
    # collect predictions here so we can print a clean confusion matrix for the best seed later
    stage3_results.append(run_experiment(**cfg, collect_predictions=True))

print_results_table(stage3_results, "Stage 3 summary")

avg_macro_f1 = np.mean([r["final_metrics"]["macro_f1"] for r in stage3_results])
avg_accuracy = np.mean([r["final_metrics"]["accuracy"] for r in stage3_results])
avg_f1_no = np.mean([r["final_metrics"]["f1_no"] for r in stage3_results])
avg_f1_maybe = np.mean([r["final_metrics"]["f1_maybe"] for r in stage3_results])
avg_f1_yes = np.mean([r["final_metrics"]["f1_yes"] for r in stage3_results])

std_macro_f1 = np.std([r["final_metrics"]["macro_f1"] for r in stage3_results])
std_accuracy = np.std([r["final_metrics"]["accuracy"] for r in stage3_results])

best_final_run = max(stage3_results, key=lambda x: x["final_metrics"]["macro_f1"])

print()
print_divider()
print("FINAL BEST MODEL CHOSEN")
print_divider()
print("Chosen config")
print(f"  model               : {model_id}")
print(f"  train batch size    : {train_batch_size}")
print(f"  eval batch size     : {eval_batch_size}")
print(f"  grad accumulation   : {best_final_run['gradient_accumulation_steps']}")
print(f"  learning rate       : {best_final_run['learning_rate']}")
print(f"  weighted loss       : {best_final_run['use_weighted_loss']}")
print(f"  weighted sampler    : {best_final_run['use_weighted_sampler']}")
print()
print("Best single seed run")
print(f"  run_name            : {best_final_run['run_name']}")
print(f"  seed                : {best_final_run['seed']}")
print(f"  accuracy            : {best_final_run['final_metrics']['accuracy']:.4f}")
print(f"  macro_f1            : {best_final_run['final_metrics']['macro_f1']:.4f}")
print(f"  f1_no               : {best_final_run['final_metrics']['f1_no']:.4f}")
print(f"  f1_maybe            : {best_final_run['final_metrics']['f1_maybe']:.4f}")
print(f"  f1_yes              : {best_final_run['final_metrics']['f1_yes']:.4f}")
print(f"  runtime             : {best_final_run['runtime_minutes']:.2f} min")
print()
print("Stage 3 mean across 3 seeds")
print(f"  avg_accuracy        : {avg_accuracy:.4f}")
print(f"  avg_macro_f1        : {avg_macro_f1:.4f}")
print(f"  avg_f1_no           : {avg_f1_no:.4f}")
print(f"  avg_f1_maybe        : {avg_f1_maybe:.4f}")
print(f"  avg_f1_yes          : {avg_f1_yes:.4f}")
print(f"  std_accuracy        : {std_accuracy:.4f}")
print(f"  std_macro_f1        : {std_macro_f1:.4f}")

print_confusion_matrix_nicely(best_final_run["confusion_matrix"])
print_divider()


STAGE 3
Final stability check. We take the best Stage 2 config and run it across 3 seeds so we can see whether it is genuinely stable or just lucky.

Run 1/3
{'eval_loss': 1.139478325843811, 'eval_accuracy': 0.34, 'eval_macro_f1': 0.29510131811922097, 'eval_f1_no': 0.15384615384615385, 'eval_f1_maybe': 0.2608695652173913, 'eval_f1_yes': 0.47058823529411764, 'eval_runtime': 20.5418, 'eval_samples_per_second': 2.434, 'eval_steps_per_second': 0.341, 'epoch': 1.0}
{'eval_loss': 1.1127160787582397, 'eval_accuracy': 0.44, 'eval_macro_f1': 0.3761256287572077, 'eval_f1_no': 0.23076923076923078, 'eval_f1_maybe': 0.3157894736842105, 'eval_f1_yes': 0.5818181818181818, 'eval_runtime': 20.9177, 'eval_samples_per_second': 2.39, 'eval_steps_per_second': 0.335, 'epoch': 2.0}
{'eval_loss': 1.0716556310653687, 'eval_accuracy': 0.44, 'eval_macro_f1': 0.3799691833590139, 'eval_f1_no': 0.2, 'eval_f1_maybe': 0.36363636363636365, 'eval_f1_yes': 0.576271186440678, 'eval_runtime': 20.4646, 'eval_samples_per_s